In [1]:
%%capture
# Installing required packages
!pip install datasets torch python-Levenshtein

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import random
import json
import math

# Checking CUDA device gpu or cpu

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch: {torch.__version__}")

c:\Users\rishi\Desktop\Projects\TransLiteration\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
PyTorch: 2.11.0+cpu


In [4]:
# Downloading dataset from huggingface

ds = load_dataset("ramgopal-reddy/Telugu_to_English_Transliteration_Dakshina")["train"]
print(f"Dataset size: {len(ds)}")
print(ds[0])


telugu_words  = ds["native_word"]
english_words = ds["english_word"]

Dataset size: 50927
{'unique_identifier': 'tel1', 'native_word': 'దుర్మార్గపు', 'english_word': 'dhurmaargapu', 'source': 'Dakshina'}


## 2. Vocabulary

In [5]:
SPECIAL_TOKENS = ['<pad>', '<sos>', '<eos>', '<unk>']
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

def build_vocab(words):
    chars = set()
    for word in words:
        chars.update(list(word))
    char2idx = {c: i + len(SPECIAL_TOKENS) for i, c in enumerate(sorted(chars))}
    for i, token in enumerate(SPECIAL_TOKENS):
        char2idx[token] = i
    idx2char = {i: c for c, i in char2idx.items()}
    return char2idx, idx2char

tel_char2idx, tel_idx2char = build_vocab(telugu_words)
eng_char2idx, eng_idx2char = build_vocab(english_words)

print(f"Telugu vocab size : {len(tel_char2idx)}")
print(f"English vocab size: {len(eng_char2idx)}")

max_tel = max(len(w) for w in telugu_words)
max_eng = max(len(w) for w in english_words)
MAX_SRC_LEN = max_eng + 2
MAX_TRG_LEN = max_tel + 2
print(f"MAX_SRC_LEN={MAX_SRC_LEN}, MAX_TRG_LEN={MAX_TRG_LEN}")

Telugu vocab size : 66
English vocab size: 30
MAX_SRC_LEN=27, MAX_TRG_LEN=22


## 3. Encoding + Data Augmentation

In [6]:
def encode(word, char2idx, max_len):
    seq = [SOS_IDX]
    for c in word:
        seq.append(char2idx.get(c, UNK_IDX))
    seq.append(EOS_IDX)
    seq = seq[:max_len]
    seq += [PAD_IDX] * (max_len - len(seq))
    return seq

def augment(word, swap_p=0.05, drop_p=0.03):
    """
    Light character-level augmentation on Telugu input.
    - swap_p: probability of swapping two adjacent characters
    - drop_p: probability of dropping a character (skip on short words)
    Applied only during training; inference gets the clean word.
    """
    chars = list(word)
    # Random adjacent swap
    if len(chars) > 2 and random.random() < swap_p:
        i = random.randint(0, len(chars) - 2)
        chars[i], chars[i + 1] = chars[i + 1], chars[i]
    # Random drop (keep at least 2 chars)
    if len(chars) > 3 and random.random() < drop_p:
        i = random.randint(0, len(chars) - 1)
        chars.pop(i)
    return ''.join(chars)

# Sanity check
print("Original:", english_words[0], "->", telugu_words[0])
print("Encoded trg:", encode(telugu_words[0], tel_char2idx, MAX_TRG_LEN))
print("Encoded src:", encode(english_words[0], eng_char2idx, MAX_SRC_LEN))

Original: dhurmaargapu -> దుర్మార్గపు
Encoded trg: [1, 35, 56, 44, 65, 42, 53, 44, 65, 21, 38, 56, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Encoded src: [1, 7, 11, 24, 21, 16, 4, 4, 21, 10, 4, 19, 24, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## 4. Dataset & DataLoader

In [7]:
class TransliterationDataset(Dataset):
    def __init__(self, eng, tel, augment_fn=None):
        self.eng = eng
        self.tel = tel
        self.augment_fn = augment_fn  # None at eval time

    def __len__(self):
        return len(self.eng)

    def __getitem__(self, idx):
        eng_word = self.eng[idx]
        if self.augment_fn is not None:
            eng_word = self.augment_fn(eng_word)
        src = torch.tensor(encode(eng_word,       eng_char2idx, MAX_SRC_LEN), dtype=torch.long)
        trg = torch.tensor(encode(self.tel[idx],  tel_char2idx, MAX_TRG_LEN), dtype=torch.long)
        return src, trg

full_ds   = TransliterationDataset( english_words, telugu_words)
val_size  = int(0.1 * len(full_ds))
train_size = len(full_ds) - val_size
train_ds, val_ds = torch.utils.data.random_split(
    full_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Augmentation only on training set
train_ds.dataset = TransliterationDataset(
    english_words, telugu_words, augment_fn=augment
)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train: {train_size} | Val: {val_size}")

Train: 45835 | Val: 5092


## 5. Model — Seq2Seq LSTM with Attention

In [ ]:
class Attention(nn.Module):
    def __init__(self, enc_hidden_dim, dec_hidden_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hidden_dim * 2 + dec_hidden_dim, dec_hidden_dim)
        self.v    = nn.Linear(dec_hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[1]
        hidden  = hidden.permute(1, 0, 2).repeat(1, src_len, 1)   # [B, T, dec_H]
        energy  = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        return torch.softmax(self.v(energy).squeeze(2), dim=1)     # [B, T]


class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_IDX)
        self.lstm      = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell   = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)
        hidden = torch.tanh(self.fc_hidden(
            torch.cat((hidden[-2], hidden[-1]), dim=1)
        )).unsqueeze(0)
        cell = torch.tanh(self.fc_cell(
            torch.cat((cell[-2], cell[-1]), dim=1)
        )).unsqueeze(0)
        return outputs, hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hidden_dim, dec_hidden_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_IDX)
        self.attention = Attention(enc_hidden_dim, dec_hidden_dim)
        self.lstm      = nn.LSTM(emb_dim + enc_hidden_dim * 2, dec_hidden_dim, batch_first=True)
        self.fc        = nn.Linear(dec_hidden_dim + enc_hidden_dim * 2 + emb_dim, output_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x, hidden, cell, encoder_outputs):
        embedded     = self.dropout(self.embedding(x.unsqueeze(1)))         # [B,1,emb]
        attn_weights = self.attention(hidden, encoder_outputs)               # [B,T]
        context      = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs) # [B,1,H*2]
        lstm_input   = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        prediction   = self.fc(
            torch.cat((output, context, embedded), dim=2).squeeze(1)
        )
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len    = trg.shape[1]
        trg_vocab  = self.decoder.fc.out_features

        outputs   = torch.zeros(batch_size, trg_len, trg_vocab).to(self.device)
        enc_out, hidden, cell = self.encoder(src)
        dec_input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(dec_input, hidden, cell, enc_out)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            dec_input = trg[:, t] if teacher_force else output.argmax(1)

        return outputs

## 6. Initialise LSTM Model

In [8]:
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
ENC_HID_DIM = 256
DEC_HID_DIM = 256
DROPOUT     = 0.5   # IMPROVEMENT: was 0.3, raised to fight overfitting

encoder = Encoder(len(eng_char2idx), ENC_EMB_DIM, ENC_HID_DIM, DROPOUT)
decoder = Decoder(len(tel_char2idx), DEC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DROPOUT)
model   = Seq2Seq(encoder, decoder, device).to(device)

def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 2,241,346


## 7. Training — with Early Stopping

In [9]:
# IMPROVEMENT: weight_decay for L2 regularisation
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

CLIP               = 1.0
N_EPOCHS           = 80
EARLY_STOP_PATIENCE = 8    # IMPROVEMENT: stop when val loss stops improving
SAVE_PATH          = "eng2tel_lstm_best.pt"

def run_epoch(loader, train=True, teacher_forcing_ratio=0.5):
    model.train() if train else model.eval()
    total_loss = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            if train:
                optimizer.zero_grad()
            output     = model(src, trg, teacher_forcing_ratio if train else 0.0)
            output_dim = output.shape[-1]
            loss = criterion(
                output[:, 1:].reshape(-1, output_dim),
                trg[:, 1:].reshape(-1)
            )
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)

best_val_loss    = float('inf')
patience_counter = 0
history          = {'train': [], 'val': []}

print("Starting training...")
print(f"{'Epoch':>6} {'TF':>5} {'Train':>8} {'Val':>8} {'LR':>10} {'Status'}")
print('-' * 60)

for epoch in range(1, N_EPOCHS + 1):
    tf_ratio   = max(0.5, 1.0 - epoch * 0.025)
    train_loss = run_epoch(train_loader, train=True,  teacher_forcing_ratio=tf_ratio)
    val_loss   = run_epoch(val_loader,   train=False)
    scheduler.step(val_loss)

    history['train'].append(train_loss)
    history['val'].append(val_loss)

    lr     = optimizer.param_groups[0]['lr']
    status = ''

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
        status = '✓ saved'
    else:
        patience_counter += 1
        status = f'patience {patience_counter}/{EARLY_STOP_PATIENCE}'

    if epoch % 5 == 0 or epoch == 1 or patience_counter >= EARLY_STOP_PATIENCE:
        print(f"{epoch:>6} {tf_ratio:>5.2f} {train_loss:>8.4f} {val_loss:>8.4f} {lr:>10.2e}  {status}")

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}. Best val loss: {best_val_loss:.4f}")
        break

print(f"\nBest val loss: {best_val_loss:.4f}")

Starting training...
 Epoch    TF    Train      Val         LR Status
------------------------------------------------------------
     1  0.97   1.7862   2.0436   1.00e-03  ✓ saved
     5  0.88   0.3426   0.7371   1.00e-03  ✓ saved
    10  0.75   0.2999   0.5479   1.00e-03  ✓ saved
    15  0.62   0.2870   0.4339   1.00e-03  ✓ saved
    20  0.50   0.2880   0.3936   1.00e-03  ✓ saved
    25  0.50   0.2555   0.3637   1.00e-03  patience 1/8
    30  0.50   0.2475   0.3692   1.00e-03  patience 1/8
    35  0.50   0.2310   0.3265   1.00e-03  ✓ saved
    40  0.50   0.1929   0.2992   5.00e-04  ✓ saved
    45  0.50   0.1805   0.2885   5.00e-04  ✓ saved
    50  0.50   0.1766   0.2953   5.00e-04  patience 2/8
    55  0.50   0.1501   0.2685   2.50e-04  patience 1/8
    60  0.50   0.1347   0.2525   1.25e-04  ✓ saved
    65  0.50   0.1341   0.2578   1.25e-04  patience 3/8
    70  0.50   0.1314   0.2625   6.25e-05  patience 3/8
    75  0.50   0.1242   0.2485   3.13e-05  patience 3/8
    80  0.50   0.1

## 8. Inference — Greedy & Beam Search

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# INFERENCE CELL
# ═══════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
import json, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

def encode(word, char2idx, max_len):
    seq = [SOS_IDX] + [char2idx.get(c, UNK_IDX) for c in word] + [EOS_IDX]
    seq = seq[:max_len]
    seq += [PAD_IDX] * (max_len - len(seq))
    return seq

# ── Model classes ─────────────────────────────────────────────────────────────
class Attention(nn.Module):
    def __init__(self, enc_hidden_dim, dec_hidden_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hidden_dim * 2 + dec_hidden_dim, dec_hidden_dim)
        self.v    = nn.Linear(dec_hidden_dim, 1, bias=False)
    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[1]
        hidden  = hidden.permute(1, 0, 2).repeat(1, src_len, 1)
        energy  = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        return torch.softmax(self.v(energy).squeeze(2), dim=1)

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_IDX)
        self.lstm      = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell   = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout   = nn.Dropout(dropout)
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)
        hidden = torch.tanh(self.fc_hidden(torch.cat((hidden[-2], hidden[-1]), dim=1))).unsqueeze(0)
        cell   = torch.tanh(self.fc_cell(torch.cat((cell[-2],   cell[-1]),   dim=1))).unsqueeze(0)
        return outputs, hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hidden_dim, dec_hidden_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_IDX)
        self.attention = Attention(enc_hidden_dim, dec_hidden_dim)
        self.lstm      = nn.LSTM(emb_dim + enc_hidden_dim * 2, dec_hidden_dim, batch_first=True)
        self.fc        = nn.Linear(dec_hidden_dim + enc_hidden_dim * 2 + emb_dim, output_dim)
        self.dropout   = nn.Dropout(dropout)
    def forward(self, x, hidden, cell, encoder_outputs):
        embedded     = self.dropout(self.embedding(x.unsqueeze(1)))
        attn_weights = self.attention(hidden, encoder_outputs)
        context      = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        lstm_input   = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        prediction   = self.fc(torch.cat((output, context, embedded), dim=2).squeeze(1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

# ── Load checkpoint — handles both full and weights-only .pt files ────────────
FULL_CKPT  = "eng2tel_lstm_full.pt"   

_HARDCODED_CFG = {
    'enc_emb_dim': 128,
    'dec_emb_dim': 128,
    'enc_hid_dim': 256,
    'dec_hid_dim': 256,
    'dropout'    : 0.5,
    'max_src_len': 27,   # max English word len + 2  (from training output)
    'max_trg_len': 22,   # max Telugu word len + 2   (from training output)
}

if os.path.exists(FULL_CKPT):
    # ── Path A: full checkpoint ───────────────────────────────────────────────
    print(f"Loading full checkpoint: {FULL_CKPT}")
    _ckpt = torch.load(FULL_CKPT, map_location=device, weights_only=False)
    eng_char2idx = _ckpt['eng_char2idx']
    tel_char2idx = _ckpt['tel_char2idx']
    tel_idx2char = {int(k): v for k, v in _ckpt['tel_idx2char'].items()}
    eng_idx2char = {int(k): v for k, v in _ckpt['eng_idx2char'].items()}
    _cfg         = _ckpt['config']
    MAX_SRC_LEN  = _cfg['max_src_len']
    MAX_TRG_LEN  = _cfg['max_trg_len']
    _state_dict  = _ckpt['model_state_dict']

else:
    raise FileNotFoundError(
        f" '{FULL_CKPT}' found.\n"
        "Make sure the .pt file is in the same folder as this notebook."
    )

# Build and load model
_encoder = Encoder(len(eng_char2idx), _cfg['enc_emb_dim'], _cfg['enc_hid_dim'], _cfg['dropout'])
_decoder = Decoder(len(tel_char2idx), _cfg['dec_emb_dim'], _cfg['enc_hid_dim'], _cfg['dec_hid_dim'], _cfg['dropout'])
model    = Seq2Seq(_encoder, _decoder, device).to(device)
model.load_state_dict(_state_dict)
model.eval()

print(f"  English vocab : {len(eng_char2idx)} chars")
print(f"  Telugu vocab  : {len(tel_char2idx)} chars")
print(f"  MAX_SRC_LEN={MAX_SRC_LEN}  MAX_TRG_LEN={MAX_TRG_LEN}")
print(f"  Device: {device}")

# ── Greedy decoding ───────────────────────────────────────────────────────────
def predict_greedy(word, max_len=30):
    model.eval()
    with torch.no_grad():
        src       = torch.tensor([encode(word, eng_char2idx, MAX_SRC_LEN)], dtype=torch.long).to(device)
        enc_out, hidden, cell = model.encoder(src)
        dec_input = torch.tensor([SOS_IDX], dtype=torch.long).to(device)
        result    = []
        for _ in range(max_len):
            output, hidden, cell = model.decoder(dec_input, hidden, cell, enc_out)
            top1 = output.argmax(1).item()
            if top1 == EOS_IDX:
                break
            if top1 not in (PAD_IDX, SOS_IDX):
                result.append(tel_idx2char.get(top1, ''))
            dec_input = torch.tensor([top1], dtype=torch.long).to(device)
        return ''.join(result)

# ── Beam search decoding ──────────────────────────────────────────────────────
def predict_beam(word, beam_size=5, max_len=30):
    model.eval()
    with torch.no_grad():
        src       = torch.tensor([encode(word, eng_char2idx, MAX_SRC_LEN)], dtype=torch.long).to(device)
        enc_out, hidden, cell = model.encoder(src)
        beams     = [(0.0, [SOS_IDX], hidden, cell)]
        completed = []
        for _ in range(max_len):
            new_beams = []
            for score, seq, h, c in beams:
                dec_in    = torch.tensor([seq[-1]], dtype=torch.long).to(device)
                out, h2, c2 = model.decoder(dec_in, h, c, enc_out)
                log_probs = F.log_softmax(out, dim=-1).squeeze(0)
                topk_vals, topk_idxs = log_probs.topk(beam_size)
                for log_p, idx in zip(topk_vals.tolist(), topk_idxs.tolist()):
                    new_seq   = seq + [idx]
                    new_score = score + log_p
                    if idx == EOS_IDX:
                        completed.append((new_score / len(new_seq), new_seq))
                    else:
                        new_beams.append((new_score, new_seq, h2, c2))
            if not new_beams:
                break
            beams = sorted(new_beams, key=lambda x: x[0] / len(x[1]), reverse=True)[:beam_size]
        if not completed:
            completed = [(s / len(seq), seq) for s, seq, *_ in beams]
        best_seq = max(completed, key=lambda x: x[0])[1]
        return ''.join(
            tel_idx2char.get(i, '')
            for i in best_seq
            if i not in (PAD_IDX, SOS_IDX, EOS_IDX)
        )

def predict(word, beam_size=5):
    return predict_beam(word, beam_size=beam_size)

# ── Test cases ────────────────────────────────────────────────────────────────
test_pairs = [
    ('దుర్మార్గపు', 'dhurmaargapu'),
    ('చూపరులకు',   'chooparulaku'),
    ('తప్పును',     'thappunu'),
    ('ముఖ్యుడు',    'mukyudu'),
    ('మైత్రి',      'maitri'),
]

print(f"\n{'English':<25} {'Greedy':<22} {'Beam(5)':<22} {'Expected':<22} {'Match'}")
print('-' * 100)
for tel, eng in test_pairs:
    greedy = predict_greedy(eng)
    beam   = predict_beam(eng)
    match  = '✓' if beam == tel else '✗'
    print(f"{eng:<25} {greedy:<22} {beam:<22} {tel:<22} {match}")

print("\n── Try your own word ──")
custom = "namasthe"
print(f"  {custom} → {predict(custom)}")


Loading full checkpoint: eng2tel_lstm_full.pt
  English vocab : 30 chars
  Telugu vocab  : 66 chars
  MAX_SRC_LEN=27  MAX_TRG_LEN=22
  Device: cpu

English                   Greedy                 Beam(5)                Expected               Match
----------------------------------------------------------------------------------------------------
dhurmaargapu              దుర్మార్గపు            దుర్మార్గపు            దుర్మార్గపు            ✓
chooparulaku              చూపరులకు               చూపరులకు               చూపరులకు               ✓
thappunu                  తప్పును                తప్పును                తప్పును                ✓
mukyudu                   ముఖ్యుడు               ముఖ్యుడు               ముఖ్యుడు               ✓
maitri                    మైత్రి                 మైత్రి                 మైత్రి                 ✓

── Try your own word ──
  namasthe → నమస్తే


In [17]:
english_words[0]

'dhurmaargapu'

In [36]:
english_len= len(testing_ds_eng)
print(english_len)

500


In [33]:
testing_ds=ds[-500::]
print(testing_ds)

{'unique_identifier': ['tel50428', 'tel50429', 'tel50430', 'tel50431', 'tel50432', 'tel50433', 'tel50434', 'tel50435', 'tel50436', 'tel50437', 'tel50438', 'tel50439', 'tel50440', 'tel50441', 'tel50442', 'tel50443', 'tel50444', 'tel50445', 'tel50446', 'tel50447', 'tel50448', 'tel50449', 'tel50450', 'tel50451', 'tel50452', 'tel50453', 'tel50454', 'tel50455', 'tel50456', 'tel50457', 'tel50458', 'tel50459', 'tel50460', 'tel50461', 'tel50462', 'tel50463', 'tel50464', 'tel50465', 'tel50466', 'tel50467', 'tel50468', 'tel50469', 'tel50470', 'tel50471', 'tel50472', 'tel50473', 'tel50474', 'tel50475', 'tel50476', 'tel50477', 'tel50478', 'tel50479', 'tel50480', 'tel50481', 'tel50482', 'tel50483', 'tel50484', 'tel50485', 'tel50486', 'tel50487', 'tel50488', 'tel50489', 'tel50490', 'tel50491', 'tel50492', 'tel50493', 'tel50494', 'tel50495', 'tel50496', 'tel50497', 'tel50498', 'tel50499', 'tel50500', 'tel50501', 'tel50502', 'tel50503', 'tel50504', 'tel50505', 'tel50506', 'tel50507', 'tel50508', 'tel5

In [35]:
testing_ds_tel=testing_ds["native_word"]
testing_ds_eng=testing_ds["english_word"]

## 9. Evaluation

In [ ]:
print(f"\n{'English':<25} {'Beam(5)':<22} {'Expected':<22} {'Match'}")
print('-' * 100)
correct=0
total=0
for i in range(english_len):
    # greedy = predict_greedy(english_words[i])
    beam   = predict(testing_ds_eng[i])
    match  = '✓' if beam == testing_ds_tel[i] else '✗'
    if beam==testing_ds_tel[i]:
        correct=correct+1
    total=total+1
    print(f"{testing_ds_eng[i]:<25} {beam:<22} {testing_ds_tel[i]:<22} {match}")




English                   Beam(5)                Expected               Match
----------------------------------------------------------------------------------------------------
merupulaa                 మెరుపులా               మెరుపులా               ✓
jeevistaaru               జీవిస్తారు             జీవిస్తారు             ✓
kantaravu                 కాంతారావు              కాంతారావు              ✓
deeshapu                  దేశపు                  దేశపు                  ✓
cheethulnii               చేతుల్నీ               చేతుల్నీ               ✓
gotic                     గోటిక్                 గోతిక్                 ✗
mathamu                   మతము                   మతము                   ✓
gamanika                  గమనిక                  గమనిక                  ✓
entho                     ఎంతో                   ఏంతో                   ✗
chuuck                    చక్                    చక్                    ✓
bhavistanu                భావిస్తాను             భావిస్తాను             ✓
koorke

In [38]:
accuracy_score_value = (correct/(total))*100
print(f"Accuracy: {accuracy_score_value} %" )

Accuracy: 91.2 %


## 13. Save Full Checkpoint for HuggingFace

In [ ]:
# Save vocabs
with open('tel_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(tel_char2idx, f, ensure_ascii=False, indent=2)
with open('eng_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(eng_char2idx, f, ensure_ascii=False, indent=2)

# Save LSTM checkpoint
checkpoint = {
    'model_state_dict': model.state_dict(),
    'tel_char2idx'    : tel_char2idx,
    'eng_char2idx'    : eng_char2idx,
    'tel_idx2char'    : {str(k): v for k, v in tel_idx2char.items()},
    'eng_idx2char'    : {str(k): v for k, v in eng_idx2char.items()},
    'config': {
        'model_type'  : 'lstm_seq2seq',
        'enc_emb_dim' : ENC_EMB_DIM,
        'dec_emb_dim' : DEC_EMB_DIM,
        'enc_hid_dim' : ENC_HID_DIM,
        'dec_hid_dim' : DEC_HID_DIM,
        'dropout'     : DROPOUT,
        'max_src_len' : MAX_SRC_LEN,
        'max_trg_len' : MAX_TRG_LEN,
        'src_lang'    : 'eng_roman',
        'trg_lang'    : 'tel',
    }
}
torch.save(checkpoint, 'eng2tel_lstm_full.pt')
print("Saved: eng2tel_lstm_full.pt, tel_vocab.json, eng_vocab.json")

Saved: eng2tel_lstm_full.pt, tel_vocab.json, eng_vocab.json
